# 17 · Citation Generation

**Idea:** grounded generation answers *only* from retrieved context and tags each sentence with its `[source]`. Citations make answers **verifiable** and failures **debuggable**: if a claim cites a source that was never retrieved, that is a hallucination signal. We generate cited answers, then validate the tags.

> Runs on Ollama. One cell optionally uses OpenAI — skips if no key.

In [1]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

In [2]:
from rag_lab import load_markdown, NumpyStore

store = NumpyStore(provider='ollama'); store.add(load_markdown())
query = 'What is reranking and why does it come after a first-stage retriever?'
hits = store.search(query, k=4)
print('retrieved sources:', sorted({(h.chunk.source, h.score) for h in hits}))

retrieved sources: [('hybrid_and_reranking.md', 0.6041040420532227), ('hybrid_and_reranking.md', 0.7177066802978516), ('rag_overview.md', 0.5922495722770691), ('systems_and_teams.md', 0.6599311828613281)]


Generate an answer with inline citations, then check every cited tag against the sources we actually retrieved.

In [3]:
from rag_lab import cited_answer
from study_buddy import CostTracker

tracker = CostTracker()
res = cited_answer(query, hits, provider='ollama', tracker=tracker)
print(res.answer)
print('\ncited:  ', res.cited)
print('valid:  ', res.valid)
print('invalid:', res.invalid, '(cited but not retrieved -> suspicious)')
print('cost:   ', tracker.summary())

[hybrid_and_reranking.md] A first-stage retriever favors recall — return a large candidate set cheaply. A
reranker then favors precision, scoring each candidate against the query with a
more expensive model. A cross-encoder reads the query and passage together for an
accurate relevance score. When you have no cross-encoder, an LLM can rerank by
scoring or ordering the candidates directly.

---

[systems_and_teams.md] Beacon is the reranking component that Atlas calls to reorder its top candidates
before they reach the generator. Beacon is a cross-encoder trained on labeled
support tickets. It was built by the Search Platform team.

---

[hybrid_and_reranking.md] Hybrid search runs both retrievers and fuses their results. Reciprocal Rank
Fusion (RRF) is a simple, robust method: each result gets a score of one over a
constant plus its rank in each list, and the scores are summed. RRF needs no
score calibration between the two systems, which makes it a strong default.

---

[rag_overview.

Stress-test grounding: ask something the corpus **cannot** answer. A well-grounded model should decline rather than invent a citation.

In [4]:
off_topic = 'What was the stock price of the company yesterday?'
res2 = cited_answer(off_topic, store.search(off_topic, k=3), provider='ollama')
print(res2.answer)
print('\ninvalid citations:', res2.invalid)

[systems_and_teams.md] The Search Platform team owns both Atlas and Beacon. The team is led by Priya Nair
and focuses on retrieval quality, indexing, and reranking across the company. It
publishes the shared embedding guidelines that other teams follow.

---

[systems_and_teams.md] Beacon is the reranking component that Atlas calls to reorder its top candidates
before they reach the generator. Beacon is a cross-encoder trained on labeled
support tickets. It was built by the Search Platform team.

---

[systems_and_teams.md] Atlas is the retrieval service that powers the Helpdesk Assistant. It indexes the
support knowledge base, embeds passages with the nomic-embed-text model, and stores
them in a Chroma vector database. Atlas was built by the Search Platform team and
calls the Beacon reranker before returning results.
ChatResponse(text="I cannot answer that question based on the provided context. The documents discuss the Search Platform team, Atlas, and Beacon systems but do not conta

Optionally generate the cited answer with OpenAI. Skips if no key.

In [5]:
try:
    cloud = CostTracker()
    r = cited_answer("What was the stock price of the company yesterday", hits, provider='openai', model='gpt-4o-mini', tracker=cloud)
    print(r.answer)
    print('\nvalid:', r.valid, '| invalid:', r.invalid)
    print('cost:', cloud.summary())
except Exception as e:
    print('openai skipped:', type(e).__name__, e)

[hybrid_and_reranking.md] A first-stage retriever favors recall — return a large candidate set cheaply. A
reranker then favors precision, scoring each candidate against the query with a
more expensive model. A cross-encoder reads the query and passage together for an
accurate relevance score. When you have no cross-encoder, an LLM can rerank by
scoring or ordering the candidates directly.

---

[systems_and_teams.md] Beacon is the reranking component that Atlas calls to reorder its top candidates
before they reach the generator. Beacon is a cross-encoder trained on labeled
support tickets. It was built by the Search Platform team.

---

[hybrid_and_reranking.md] Hybrid search runs both retrievers and fuses their results. Reciprocal Rank
Fusion (RRF) is a simple, robust method: each result gets a score of one over a
constant plus its rank in each list, and the scores are summed. RRF needs no
score calibration between the two systems, which makes it a strong default.

---

[rag_overview.

**Takeaway:** citations turn a RAG answer into a *checkable* claim. Validating cited tags against retrieved sources is a cheap, automatic hallucination check — the seed of the evaluation harness in notebook 21 and the capstone.

That closes the Frontier RAG block (13–17). Next up in the roadmap → `18_document_ingestion.ipynb`.